In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ---------------------------------------------------------
# 1. Load Required Tables (Silver Layer)
# ---------------------------------------------------------
# Course catalog (no silver equivalent exists, using bronze as dimension table)
course_catalog_df = spark.read.table("edtech_project.edtech_bronze.course_catalog_bronze")

# Silver layer tables
engagement_df = spark.read.table("edtech_project.edtech_silver.student_course_engagement")
risk_profile_df = spark.read.table("edtech_project.edtech_silver.enrollment_risk_profile")

# ---------------------------------------------------------
# 2. Derive Course-Level Performance Metrics
# ---------------------------------------------------------
active_per_course = engagement_df \
    .groupBy("course_id") \
    .agg(F.countDistinct("student_id").alias("active_students"))

completion_per_course = engagement_df.groupBy("course_id") \
    .agg(F.avg("video_completion_rate").alias("completion_rate"))

quiz_per_course = engagement_df.filter(F.col("quiz_attempts_count") > 0) \
    .groupBy("course_id") \
    .agg(F.avg("quiz_pass_rate").alias("avg_quiz_score"))

course_perf_df = active_per_course \
    .join(completion_per_course, "course_id", "outer") \
    .join(quiz_per_course, "course_id", "outer")

# ---------------------------------------------------------
# 3. Calculate Base Metrics per Instructor
# ---------------------------------------------------------
instructor_base_metrics = course_catalog_df.join(course_perf_df, "course_id", "inner") \
    .groupBy("instructor_id", "instructor_name") \
    .agg(
        F.countDistinct("course_id").alias("active_courses"),
        F.sum("active_students").alias("total_active_students"),
        F.avg("completion_rate").alias("avg_completion_rate"),
        F.avg("avg_quiz_score").alias("avg_student_satisfaction")
    )

# ---------------------------------------------------------
# 4. High-Risk Students per Instructor
# ---------------------------------------------------------
high_risk_df = risk_profile_df.filter(F.col("risk_tier") == "HIGH") \
    .join(course_catalog_df, "course_id", "inner") \
    .groupBy("instructor_id") \
    .agg(F.countDistinct("student_id").alias("students_at_high_dropout_risk"))

# ---------------------------------------------------------
# 5. Combine and Rank
# ---------------------------------------------------------
rank_window = Window.orderBy(F.col("avg_completion_rate").desc())
current_week = F.date_trunc("week", F.current_date())

current_week_df = instructor_base_metrics \
    .join(high_risk_df, "instructor_id", "left") \
    .fillna({"students_at_high_dropout_risk": 0}) \
    .withColumn("performance_rank", F.rank().over(rank_window)) \
    .withColumn("report_week", current_week)

# ---------------------------------------------------------
# 6. Historical Trend: Current Week vs 4-Week Average
# ---------------------------------------------------------
try:
    historical_df = spark.read.table("edtech_project.edtech_gold.instructor_dashboard")
    # Get prior 4 weeks (exclude current week)
    four_weeks_ago = F.date_sub(current_week, 28)
    hist_avg = historical_df \
        .filter(
            (F.col("report_week") >= four_weeks_ago) &
            (F.col("report_week") < current_week)
        ) \
        .groupBy("instructor_id") \
        .agg(
            F.avg("avg_completion_rate").alias("avg_completion_rate_4w"),
            F.avg("avg_student_satisfaction").alias("avg_satisfaction_4w"),
            F.avg("students_at_high_dropout_risk").alias("avg_high_risk_4w")
        )

    # Join trend and compute deltas
    current_week_df = current_week_df \
        .join(hist_avg, "instructor_id", "left") \
        .withColumn("completion_rate_4w_avg", F.col("avg_completion_rate_4w")) \
        .withColumn("satisfaction_4w_avg", F.col("avg_satisfaction_4w")) \
        .withColumn("high_risk_4w_avg", F.col("avg_high_risk_4w")) \
        .withColumn(
            "completion_trend",
            F.when(F.col("avg_completion_rate_4w").isNull(), F.lit("NEW"))
             .when(F.col("avg_completion_rate") > F.col("avg_completion_rate_4w") * 1.02, F.lit("IMPROVING"))
             .when(F.col("avg_completion_rate") < F.col("avg_completion_rate_4w") * 0.98, F.lit("DECLINING"))
             .otherwise(F.lit("STABLE"))
        ) \
        .drop("avg_completion_rate_4w", "avg_satisfaction_4w", "avg_high_risk_4w")

except Exception:
    # First run — no historical data yet
    current_week_df = current_week_df \
        .withColumn("completion_rate_4w_avg", F.lit(None).cast("double")) \
        .withColumn("satisfaction_4w_avg", F.lit(None).cast("double")) \
        .withColumn("high_risk_4w_avg", F.lit(None).cast("double")) \
        .withColumn("completion_trend", F.lit("NEW"))

# ---------------------------------------------------------
# 7. Write to Gold Layer (append for historical tracking)
# ---------------------------------------------------------
# Remove current week's data if re-running (idempotent)
try:
    from delta.tables import DeltaTable
    gold_table = DeltaTable.forName(spark, "edtech_project.edtech_gold.instructor_dashboard")
    gold_table.delete(F.col("report_week") == F.date_trunc("week", F.current_date()))
except Exception:
    pass  # Table doesn't exist yet

current_week_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("edtech_project.edtech_gold.instructor_dashboard")

display(current_week_df)